### Libraries

In [2]:
import pandas as pd
import numpy as np
import os
import joblib
import yaml

import warnings
warnings.filterwarnings('ignore')

### Step 1: Load Train, Val, Test Splits

In [3]:
config_file = os.path.join('..', 'config', 'config.yaml')

# Load config
with open(config_file, "r") as f:
    config = yaml.safe_load(f)

root_dir = (os.path.dirname(os.path.dirname(os.path.abspath(config_file))))
splits_path = os.path.join(root_dir, config['paths']['splits_data'])

print("The Data Splits will be saved to:", splits_path)

The Data Splits will be saved to: c:\Users\DELL\Desktop\loan-risk-system\data/splits/


In [4]:
# Classification datasets splits
X_train_cls = pd.read_csv(os.path.join(splits_path, 'X_train_cls.csv'))
X_test_cls = pd.read_csv(os.path.join(splits_path, 'X_test_cls.csv'))
X_val_cls = pd.read_csv(os.path.join(splits_path, 'X_val_cls.csv'))
y_train_cls = pd.read_csv(os.path.join(splits_path, 'y_train_cls.csv'))
y_test_cls = pd.read_csv(os.path.join(splits_path, 'y_test_cls.csv'))
y_val_cls = pd.read_csv(os.path.join(splits_path, 'y_val_cls.csv'))

# Regression datasets splits
X_train_reg = pd.read_csv(os.path.join(splits_path, 'X_train_reg.csv'))
X_test_reg = pd.read_csv(os.path.join(splits_path, 'X_test_reg.csv'))
X_val_reg = pd.read_csv(os.path.join(splits_path, 'X_val_reg.csv'))
y_train_reg = pd.read_csv(os.path.join(splits_path, 'y_train_reg.csv'))
y_test_reg = pd.read_csv(os.path.join(splits_path, 'y_test_reg.csv'))
y_val_reg = pd.read_csv(os.path.join(splits_path, 'y_val_reg.csv'))

print("Data splits loaded successfully!")
print("Classification splits shapes:" , X_train_cls.shape, X_test_cls.shape, X_val_cls.shape, y_train_cls.shape, y_test_cls.shape, y_val_cls.shape)
print("Regression splits shapes:" , X_train_reg.shape, X_test_reg.shape, X_val_reg.shape, y_train_reg.shape, y_test_reg.shape, y_val_reg.shape)

Data splits loaded successfully!
Classification splits shapes: (27693, 11) (2444, 11) (2444, 11) (27693, 1) (2444, 1) (2444, 1)
Regression splits shapes: (25045, 11) (2210, 11) (2210, 11) (25045, 1) (2210, 1) (2210, 1)


### Step 2: Remove Duplicates

In [5]:
# For Classification:
X_train_cls_dup = X_train_cls.copy()
y_train_cls_dup = y_train_cls.copy()

index_values = X_train_cls_dup[X_train_cls_dup.duplicated()].index.tolist()

print("Shape of X_train_cls_dup before removing duplicates:", X_train_cls_dup.shape)
print("Shape of y_train_cls_dup before removing duplicates:", y_train_cls_dup.shape)

X_train_cls_dup = X_train_cls_dup.drop(index=index_values)
y_train_cls_dup = y_train_cls_dup.drop(index=index_values)

print("Shape of X_train_cls_dup after removing duplicates:", X_train_cls_dup.shape)
print("Shape of y_train_cls_dup after removing duplicates:", y_train_cls_dup.shape)

print("Number of duplicate rows removed:", len(index_values))


Shape of X_train_cls_dup before removing duplicates: (27693, 11)
Shape of y_train_cls_dup before removing duplicates: (27693, 1)
Shape of X_train_cls_dup after removing duplicates: (27577, 11)
Shape of y_train_cls_dup after removing duplicates: (27577, 1)
Number of duplicate rows removed: 116


In [6]:
# For Regression:
X_train_reg_dup = X_train_reg.copy()
y_train_reg_dup = y_train_reg.copy()

index_values = X_train_reg_dup[X_train_reg_dup.duplicated()].index.tolist()

print("Shape of X_train_reg_dup before removing duplicates:", X_train_reg_dup.shape)
print("Shape of y_train_reg_dup before removing duplicates:", y_train_reg_dup.shape)

X_train_reg_dup = X_train_reg_dup.drop(index=index_values)
y_train_reg_dup = y_train_reg_dup.drop(index=index_values)

print("Shape of X_train_reg_dup after removing duplicates:", X_train_reg_dup.shape)
print("Shape of y_train_reg_dup after removing duplicates:", y_train_reg_dup.shape)

print("Number of duplicate rows removed:", len(index_values))

Shape of X_train_reg_dup before removing duplicates: (25045, 11)
Shape of y_train_reg_dup before removing duplicates: (25045, 1)
Shape of X_train_reg_dup after removing duplicates: (24943, 11)
Shape of y_train_reg_dup after removing duplicates: (24943, 1)
Number of duplicate rows removed: 102


### Step 3: Fix Cross Column Logic Violations

In [7]:
def fix_cross_column(df):
    df = df.copy()

    # Rule 1: Cap emp_length to (age - 18)
    # person_emp_length = min(person_emp_length, person_age - 18)]
    df['person_emp_length'] = df.apply(lambda row: min(row['person_emp_length'], row['person_age'] - 18) 
    if pd.notna(row['person_emp_length']) else row['person_emp_length'], axis=1)
    print("Applied Rule 1: Capped emp_length to (age - 18)")
    
    # Rule 2: Cap cred_hist_length to (age - 18)
    # cb_person_cred_hist_length = min(cb_person_cred_hist_length, person_age - 18)
    df['cb_person_cred_hist_length'] = df.apply(lambda row: min(row['cb_person_cred_hist_length'], row['person_age'] - 18) 
    if pd.notna(row['cb_person_cred_hist_length']) else row['cb_person_cred_hist_length'], axis=1)
    print("Applied Rule 2: Capped cred_hist_length to (age - 18)")

    # Rule 3: Recalculate loan_percent_income
    # loan_percent_income = loan_amnt / person_income
    df['loan_percent_income'] = df['loan_amnt'] / (df['person_income'] + 1)  # +1 to avoid division by zero
    print("Applied Rule 3: Recalculated loan_percent_income")

    return df

In [8]:
X_train_cls_fixed = X_train_cls_dup.copy()
X_test_cls_fixed = X_test_cls.copy()
X_val_cls_fixed = X_val_cls.copy()

X_train_reg_fixed = X_train_reg_dup.copy()
X_test_reg_fixed = X_test_reg.copy()
X_val_reg_fixed = X_val_reg.copy()

In [9]:
X_train_cls_fixed = fix_cross_column(X_train_cls_fixed)
print("Finished fixing X_train_cls_fixed \n")
X_test_cls_fixed = fix_cross_column(X_test_cls_fixed)
print("Finished fixing X_test_cls_fixed \n")
X_val_cls_fixed = fix_cross_column(X_val_cls_fixed)
print("Finished fixing X_val_cls_fixed \n")

X_train_reg_fixed = fix_cross_column(X_train_reg_fixed)
print("Finished fixing X_train_reg_fixed \n")
X_test_reg_fixed = fix_cross_column(X_test_reg_fixed)
print("Finished fixing X_test_reg_fixed \n")
X_val_reg_fixed = fix_cross_column(X_val_reg_fixed)
print("Finished fixing X_val_reg_fixed \n")

Applied Rule 1: Capped emp_length to (age - 18)
Applied Rule 2: Capped cred_hist_length to (age - 18)
Applied Rule 3: Recalculated loan_percent_income
Finished fixing X_train_cls_fixed 

Applied Rule 1: Capped emp_length to (age - 18)
Applied Rule 2: Capped cred_hist_length to (age - 18)
Applied Rule 3: Recalculated loan_percent_income
Finished fixing X_test_cls_fixed 

Applied Rule 1: Capped emp_length to (age - 18)
Applied Rule 2: Capped cred_hist_length to (age - 18)
Applied Rule 3: Recalculated loan_percent_income
Finished fixing X_val_cls_fixed 

Applied Rule 1: Capped emp_length to (age - 18)
Applied Rule 2: Capped cred_hist_length to (age - 18)
Applied Rule 3: Recalculated loan_percent_income
Finished fixing X_train_reg_fixed 

Applied Rule 1: Capped emp_length to (age - 18)
Applied Rule 2: Capped cred_hist_length to (age - 18)
Applied Rule 3: Recalculated loan_percent_income
Finished fixing X_test_reg_fixed 

Applied Rule 1: Capped emp_length to (age - 18)
Applied Rule 2: Cappe

### Step 4: Handle Missing Values

In [10]:
from sklearn.impute import SimpleImputer

# train
X_train_cls_miss = X_train_cls_fixed.copy()
X_train_reg_miss = X_train_reg_fixed.copy()

# test and val
X_test_cls_miss = X_test_cls_fixed.copy()
X_val_cls_miss = X_val_cls_fixed.copy()
X_test_reg_miss = X_test_reg_fixed.copy()
X_val_reg_miss = X_val_reg_fixed.copy()


# classification
simple_imputer_cls = SimpleImputer(strategy='median')
# fir_transform on train, then transform on test and val to avoid data leakage
X_train_cls_miss[['person_emp_length', 'loan_int_rate']] = simple_imputer_cls.fit_transform(X_train_cls_miss[['person_emp_length', 'loan_int_rate']])
X_test_cls_miss[['person_emp_length', 'loan_int_rate']] = simple_imputer_cls.transform(X_test_cls_miss[['person_emp_length', 'loan_int_rate']])
X_val_cls_miss[['person_emp_length', 'loan_int_rate']] = simple_imputer_cls.transform(X_val_cls_miss[['person_emp_length', 'loan_int_rate']])

# Regression
simple_imputer_reg = SimpleImputer(strategy='median')
# fir_transform on train, then transform on test and val to avoid data leakage
X_train_reg_miss[['person_emp_length']] = simple_imputer_reg.fit_transform(X_train_reg_miss[['person_emp_length']])
X_test_reg_miss[['person_emp_length']] = simple_imputer_reg.transform(X_test_reg_miss[['person_emp_length']])
X_val_reg_miss[['person_emp_length']] = simple_imputer_reg.transform(X_val_reg_miss[['person_emp_length']])

print("Cls after imputation: " , X_train_cls_miss.isnull().sum().sum())
print("Reg after imputation: " , X_train_reg_miss.isnull().sum().sum())

Cls after imputation:  0
Reg after imputation:  0


### Step 5: Handle Outlier - IQR Capping

In [11]:
X_train_cls_out = X_train_cls_miss.copy()
X_test_cls_out = X_test_cls_miss.copy()
X_val_cls_out = X_val_cls_miss.copy()

X_train_reg_out = X_train_reg_miss.copy()
X_test_reg_out = X_test_reg_miss.copy()
X_val_reg_out = X_val_reg_miss.copy()

In [12]:
# Function 1 — fit on train only
def fit_iqr_bounds(df_train):
    bounds = {}
    numeric_cols = df_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
    for col in numeric_cols:
        Q1 = df_train[col].quantile(0.25)
        Q3 = df_train[col].quantile(0.75)
        IQR = Q3 - Q1
        bounds[col] = {
            'lower': Q1 - 1.5 * IQR,
            'upper': Q3 + 1.5 * IQR
        }
    return bounds

# Function 2 — apply to any split
def apply_iqr_capping(df, bounds):
    df = df.copy()
    for col, bound in bounds.items():
        if col in df.columns:
            df[col] = df[col].clip(lower=bound['lower'], upper=bound['upper'])
    return df

In [13]:
Train_cls_bounds = fit_iqr_bounds(X_train_cls_out)
X_train_cls_out = apply_iqr_capping(X_train_cls_out, Train_cls_bounds)
X_test_cls_out = apply_iqr_capping(X_test_cls_out, Train_cls_bounds)
X_val_cls_out = apply_iqr_capping(X_val_cls_out, Train_cls_bounds)

Train_reg_bounds = fit_iqr_bounds(X_train_reg_out)
X_train_reg_out = apply_iqr_capping(X_train_reg_out, Train_reg_bounds)
X_test_reg_out = apply_iqr_capping(X_test_reg_out, Train_reg_bounds)
X_val_reg_out = apply_iqr_capping(X_val_reg_out, Train_reg_bounds)

In [14]:
# verification cell
print("Before capping:")
print(f"person_age max: {X_train_cls_miss['person_age'].max()}")
print(f"person_income max: {X_train_cls_miss['person_income'].max()}")

print("\nAfter capping:")
print(f"person_age max: {X_train_cls_out['person_age'].max()}")
print(f"person_income max: {X_train_cls_out['person_income'].max()}")

Before capping:
person_age max: 144
person_income max: 6000000

After capping:
person_age max: 40.5
person_income max: 139750


In [15]:
# Verify value ranges
print(f"person_age range: {X_train_cls_out['person_age'].min():.1f} - {X_train_cls_out['person_age'].max():.1f}")
print(f"person_income range: {X_train_cls_out['person_income'].min():.1f} - {X_train_cls_out['person_income'].max():.1f}")
print(f"person_emp_length range: {X_train_cls_out['person_emp_length'].min():.1f} - {X_train_cls_out['person_emp_length'].max():.1f}")
print(f"cb_person_cred_hist_length range: {X_train_cls_out['cb_person_cred_hist_length'].min():.1f} - {X_train_cls_out['cb_person_cred_hist_length'].max():.1f}")
print(f"loan_percent_income range: {X_train_cls_out['loan_percent_income'].min():.4f} - {X_train_cls_out['loan_percent_income'].max():.4f}")

person_age range: 20.0 - 40.5
person_income range: 4000.0 - 139750.0
person_emp_length range: 0.0 - 12.0
cb_person_cred_hist_length range: 2.0 - 15.5
loan_percent_income range: 0.0008 - 0.4368


###  Step 6: Log Transformation

Why log transformation?

person_income skewness = 35.69 — extremely skewed  
Log transform reduces skewness and helps models perform better

In [16]:
# this is my result before apply capping for skewness
'''
person_age                     2.705253
person_income                 35.694767
person_emp_length              2.837094
loan_amnt                      1.192506
loan_int_rate                  0.199957
loan_percent_income            1.065126
cb_person_cred_hist_length     1.663755
loan_status                    1.364801
'''

'\nperson_age                     2.705253\nperson_income                 35.694767\nperson_emp_length              2.837094\nloan_amnt                      1.192506\nloan_int_rate                  0.199957\nloan_percent_income            1.065126\ncb_person_cred_hist_length     1.663755\nloan_status                    1.364801\n'

In [17]:
'''
Capping reduces skewness significantly. person_age and person_emp_length are likely acceptable after capping.
But person_income was 35.69 — so extreme that capping alone is not enough.
Rule of thumb:

Skew < 1.0 after capping → no log needed
Skew > 1.0 after capping → apply log
'''

'\nCapping reduces skewness significantly. person_age and person_emp_length are likely acceptable after capping.\nBut person_income was 35.69 — so extreme that capping alone is not enough.\nRule of thumb:\n\nSkew < 1.0 after capping → no log needed\nSkew > 1.0 after capping → apply log\n'

In [18]:
print("Classification:\n",X_train_cls_out.select_dtypes(include=['int64', 'float64']).skew())
print()
print("Regression:\n",X_train_reg_out.select_dtypes(include=['int64', 'float64']).skew())

Classification:
 person_age                    1.015975
person_income                 0.867167
person_emp_length             0.790442
loan_amnt                     0.785433
loan_int_rate                 0.199678
loan_percent_income           0.801530
cb_person_cred_hist_length    1.127058
dtype: float64

Regression:
 person_age                    1.019966
person_income                 0.863614
person_emp_length             0.779735
loan_amnt                     0.811065
loan_status                   0.000000
loan_percent_income           0.812467
cb_person_cred_hist_length    1.121246
dtype: float64


In [19]:
'''
Does capping improve skewness?
Yes it does! Capping removes extreme values which are the main cause of skewness.

Conclusion:
Capping fixed almost everything 
No column has extreme skewness anymore
person_income went from 35.69 → 0.87 — capping did the job
Only person_age (1.02) and cb_person_cred_hist_length (1.13) are borderline

Decision:

Skip log transformation entirely — not needed
All skewness values are below or near 1.0
This keeps the data more interpretable and avoids unnecessary transformations.
'''

# But if even after capping, if we had a column with skew > 1.0, we would apply log transformation to that column
# in all splits (train, test, val) using the same log function fitted on train to avoid data leakage.

'\nDoes capping improve skewness?\nYes it does! Capping removes extreme values which are the main cause of skewness.\n\nConclusion:\nCapping fixed almost everything \nNo column has extreme skewness anymore\nperson_income went from 35.69 → 0.87 — capping did the job\nOnly person_age (1.02) and cb_person_cred_hist_length (1.13) are borderline\n\nDecision:\n\nSkip log transformation entirely — not needed\nAll skewness values are below or near 1.0\nThis keeps the data more interpretable and avoids unnecessary transformations.\n'

### Step 7: Ordinal Encoding

In [20]:
X_train_cls_oe = X_train_cls_out.copy()
X_test_cls_oe = X_test_cls_out.copy()
X_val_cls_oe = X_val_cls_out.copy()

X_train_reg_oe = X_train_reg_out.copy()
X_test_reg_oe = X_test_reg_out.copy()
X_val_reg_oe = X_val_reg_out.copy()

In [21]:
from sklearn.preprocessing import OrdinalEncoder

grade_order = [['A', 'B', 'C', 'D', 'E', 'F', 'G']]

# Classification
ordinal_encoder_cls = OrdinalEncoder(categories=grade_order)

X_train_cls_oe['loan_grade'] = ordinal_encoder_cls.fit_transform(X_train_cls_oe[["loan_grade"]])
X_test_cls_oe['loan_grade'] = ordinal_encoder_cls.transform(X_test_cls_oe[["loan_grade"]])
X_val_cls_oe['loan_grade'] = ordinal_encoder_cls.transform(X_val_cls_oe[["loan_grade"]])

# Regression
ordinal_encoder_reg = OrdinalEncoder(categories=grade_order)

X_train_reg_oe['loan_grade'] = ordinal_encoder_reg.fit_transform(X_train_reg_oe[["loan_grade"]])
X_test_reg_oe['loan_grade'] = ordinal_encoder_reg.transform(X_test_reg_oe[["loan_grade"]])
X_val_reg_oe['loan_grade'] = ordinal_encoder_reg.transform(X_val_reg_oe[["loan_grade"]])

In [22]:
# verification cell
print("loan_grade unique values after encoding:")
print(X_train_cls_oe['loan_grade'].unique())

loan_grade unique values after encoding:
[1. 0. 2. 3. 4. 5. 6.]


### Step 8: One-Hot Encoding

In [23]:
X_train_cls_ohe = X_train_cls_oe.copy()
X_test_cls_ohe = X_test_cls_oe.copy()
X_val_cls_ohe = X_val_cls_oe.copy()

X_train_reg_ohe = X_train_reg_oe.copy()
X_test_reg_ohe = X_test_reg_oe.copy()
X_val_reg_ohe = X_val_reg_oe.copy()

In [24]:
from sklearn.preprocessing import OneHotEncoder
ohe_cols = ['person_home_ownership', 'loan_intent']

# Classification
ohe_cls = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Fit transform train
encoded_train = pd.DataFrame(
    ohe_cls.fit_transform(X_train_cls_oe[ohe_cols]),
    columns=ohe_cls.get_feature_names_out(),
    index=X_train_cls_oe.index)

# Transform val and test
encoded_val = pd.DataFrame(
    ohe_cls.transform(X_val_cls_oe[ohe_cols]),
    columns=ohe_cls.get_feature_names_out(),
    index=X_val_cls_oe.index)

encoded_test = pd.DataFrame(
    ohe_cls.transform(X_test_cls_oe[ohe_cols]),
    columns=ohe_cls.get_feature_names_out(),
    index=X_test_cls_oe.index)

# Drop original and concat encoded
X_train_cls_ohe = pd.concat([X_train_cls_oe.drop(columns=ohe_cols), encoded_train], axis=1)
X_val_cls_ohe   = pd.concat([X_val_cls_oe.drop(columns=ohe_cols), encoded_val], axis=1)
X_test_cls_ohe  = pd.concat([X_test_cls_oe.drop(columns=ohe_cols), encoded_test], axis=1)

print(f"Shape after OHE: {X_train_cls_ohe.shape}")

Shape after OHE: (27577, 19)


In [25]:
ohe_cols = ['person_home_ownership', 'loan_intent']

# Regression
ohe_reg = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Fit transform train
encoded_train = pd.DataFrame(
    ohe_reg.fit_transform(X_train_reg_oe[ohe_cols]),
    columns=ohe_reg.get_feature_names_out(),
    index=X_train_reg_oe.index)

# Transform val and test
encoded_val = pd.DataFrame(
    ohe_reg.transform(X_val_reg_oe[ohe_cols]),
    columns=ohe_reg.get_feature_names_out(),
    index=X_val_reg_oe.index)

encoded_test = pd.DataFrame(
    ohe_reg.transform(X_test_reg_oe[ohe_cols]),
    columns=ohe_reg.get_feature_names_out(),
    index=X_test_reg_oe.index)

# Drop original and concat encoded
X_train_reg_ohe = pd.concat([X_train_reg_oe.drop(columns=ohe_cols), encoded_train], axis=1)
X_val_reg_ohe   = pd.concat([X_val_reg_oe.drop(columns=ohe_cols), encoded_val], axis=1)
X_test_reg_ohe  = pd.concat([X_test_reg_oe.drop(columns=ohe_cols), encoded_test], axis=1)

print(f"Shape after OHE: {X_train_reg_ohe.shape}")

Shape after OHE: (24943, 19)


### Step 9: Map Encoding

Label encoding only be done on target cols

In [26]:
X_train_cls_le = X_train_cls_ohe.copy()
X_test_cls_le = X_test_cls_ohe.copy()
X_val_cls_le = X_val_cls_ohe.copy()

X_train_reg_le = X_train_reg_ohe.copy()
X_test_reg_le = X_test_reg_ohe.copy()
X_val_reg_le = X_val_reg_ohe.copy()

In [27]:
# Apply to all splits — no fitting needed
# This is a fixed business rule not a learned transformation

for df in [X_train_cls_le, X_val_cls_le, X_test_cls_le,
           X_train_reg_le, X_val_reg_le, X_test_reg_le]:
    df['cb_person_default_on_file'] = df['cb_person_default_on_file'].map({'Y': 1, 'N': 0})

In [28]:
# verification cell
print("cb_person_default_on_file unique values after encoding:")
print(X_train_cls_le['cb_person_default_on_file'].unique())

cb_person_default_on_file unique values after encoding:
[0 1]


### Step 10: Feature Scaling

In [29]:
X_train_cls_scale = X_train_cls_le.copy()
X_test_cls_scale = X_test_cls_le.copy()
X_val_cls_scale = X_val_cls_le.copy()

X_train_reg_scale = X_train_reg_le.copy()
X_test_reg_scale = X_test_reg_le.copy()
X_val_reg_scale = X_val_reg_le.copy()

In [30]:
from sklearn.preprocessing import StandardScaler

# Classification
numeric_cols_cls = ['person_age', 'person_income', 'person_emp_length',
                    'loan_amnt', 'loan_int_rate', 'loan_percent_income',
                    'cb_person_cred_hist_length','loan_grade']  
scaler_cls = StandardScaler()

X_train_cls_scale[numeric_cols_cls] = scaler_cls.fit_transform(X_train_cls_scale[numeric_cols_cls])
X_val_cls_scale[numeric_cols_cls]   = scaler_cls.transform(X_val_cls_scale[numeric_cols_cls])
X_test_cls_scale[numeric_cols_cls]  = scaler_cls.transform(X_test_cls_scale[numeric_cols_cls])


# Regression
numerical_cols_reg = ['person_age','person_income', 
    'person_emp_length','loan_grade',        # ordinal encoded — scale it
    'loan_amnt','loan_percent_income','cb_person_cred_hist_length']
scaler_reg = StandardScaler()

X_train_reg_scale[numerical_cols_reg] = scaler_reg.fit_transform(X_train_reg_scale[numerical_cols_reg])
X_val_reg_scale[numerical_cols_reg]   = scaler_reg.transform(X_val_reg_scale[numerical_cols_reg])
X_test_reg_scale[numerical_cols_reg]  = scaler_reg.transform(X_test_reg_scale[numerical_cols_reg])


In [31]:
print("Before scaling:")
print(f"person_income mean: {X_train_cls_le['person_income'].mean():.4f}")
print(f"person_income std:  {X_train_cls_le['person_income'].std():.4f}")
print()
print("After scaling:")
print(f"person_income mean: {X_train_cls_scale['person_income'].mean():.4f}")
print(f"person_income std:  {X_train_cls_scale['person_income'].std():.4f}")

Before scaling:
person_income mean: 62344.0862
person_income std:  31723.6131

After scaling:
person_income mean: -0.0000
person_income std:  1.0000


### Step Extra: Making small and affective changes

In [32]:
# drop person_home_ownership_OTHER column

# Simply drop the OTHER column
# It has very low frequency and adds noise

for df in [X_train_cls_scale, X_val_cls_scale, X_test_cls_scale,
           X_train_reg_scale, X_val_reg_scale, X_test_reg_scale]:
    if 'person_home_ownership_OTHER' in df.columns:
        df.drop(columns=['person_home_ownership_OTHER'], inplace=True)

print("✅ Rare category column dropped")
print(f"New shape: {X_train_cls_scale.shape}")
print(f"New shape: {X_train_reg_scale.shape}")

✅ Rare category column dropped
New shape: (27577, 18)
New shape: (24943, 18)


In [33]:
# drop: cb_person_cred_hist_length

# Drop cb_person_cred_hist_length from all splits
# It is highly correlated with person_age (0.85)

col_to_drop = 'cb_person_cred_hist_length'

for df in [X_train_cls_scale, X_val_cls_scale, X_test_cls_scale,
           X_train_reg_scale, X_val_reg_scale, X_test_reg_scale]:
    if col_to_drop in df.columns:
        df.drop(columns=[col_to_drop], inplace=True)
        

print("✅ Redundant feature dropped")
print(f"New shape: {X_train_cls_scale.shape}")
print(f"New shape: {X_train_reg_scale.shape}")

✅ Redundant feature dropped
New shape: (27577, 17)
New shape: (24943, 17)


In [33]:
# Data Type Optimization

def optimize_dtypes(df):
    df = df.copy()
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype('float32')
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = df[col].astype('int32')
    return df


def memory_usage_mb(df):
    return df.memory_usage(deep=True).sum() / (1024 ** 2)


datasets = [
    X_train_cls_scale, X_val_cls_scale, X_test_cls_scale,
    X_train_reg_scale, X_val_reg_scale, X_test_reg_scale
]


for i in range(len(datasets)):
    before = memory_usage_mb(datasets[i])
    
    datasets[i] = optimize_dtypes(datasets[i])
    
    after = memory_usage_mb(datasets[i])
    
    saved_percent = ((before - after) / before) * 100
    
    print(f"\nDataset {i+1}")
    print(f"Before: {before:.2f} MB")
    print(f"After:  {after:.2f} MB")
    print(f"Memory saved: {saved_percent:.2f}%")
X_train_cls_scale, X_val_cls_scale, X_test_cls_scale, \
X_train_reg_scale, X_val_reg_scale, X_test_reg_scale = datasets


Dataset 1
Before: 3.79 MB
After:  2.00 MB
Memory saved: 47.22%

Dataset 2
Before: 0.32 MB
After:  0.16 MB
Memory saved: 49.98%

Dataset 3
Before: 0.32 MB
After:  0.16 MB
Memory saved: 49.98%

Dataset 4
Before: 3.43 MB
After:  1.81 MB
Memory saved: 47.22%

Dataset 5
Before: 0.29 MB
After:  0.14 MB
Memory saved: 49.98%

Dataset 6
Before: 0.29 MB
After:  0.14 MB
Memory saved: 49.98%


### Step 11: Verify Preprocessed Data

In [35]:
X_train_cls_verify = X_train_cls_scale.copy()
X_test_cls_verify = X_test_cls_scale.copy()
X_val_cls_verify = X_val_cls_scale.copy()

X_train_reg_verify = X_train_reg_scale.copy()
X_test_reg_verify = X_test_reg_scale.copy()
X_val_reg_verify = X_val_reg_scale.copy()

In [42]:
# Classification    
print("X_train_cls shape: ",X_train_cls_verify.shape)
print("X_test_cls shape: ",X_test_cls_verify.shape)
print("X_val_cls shape: ",X_val_cls_verify.shape)

print()

print("X_train_cls missing values: ",X_train_cls_verify.isnull().sum().sum())
print("X_test_cls missing values: ",X_test_cls_verify.isnull().sum().sum())
print("X_val_cls missing values: ",X_val_cls_verify.isnull().sum().sum())

print()

print("X_train_cls categorical columns: ",X_train_cls_verify.select_dtypes(include=['object']).columns.tolist())
print("X_test_cls categorical columns: ",X_test_cls_verify.select_dtypes(include=['object']).columns.tolist())
print("X_val_cls categorical columns: ",X_val_cls_verify.select_dtypes(include=['object']).columns.tolist())

print()

print("X_train_cls data types:\n",X_train_cls_verify.dtypes)
print("X_test_cls data types:\n",X_test_cls_verify.dtypes)
print("X_val_cls data types:\n",X_val_cls_verify.dtypes)

print()

print("y_train_cls distribution: ",y_train_cls_dup["loan_status"].value_counts(normalize=True))

print()

# Check 1: Infinite values
print("Infinite values in X_train_cls:", np.isinf(X_train_cls_verify).sum().sum())
print("Infinite values in X_test_cls:", np.isinf(X_test_cls_verify).sum().sum())
print("Infinite values in X_val_cls:", np.isinf(X_val_cls_verify).sum().sum())

print()
# Check 2: Constant columns
for col in [X_train_cls_verify,X_test_cls_verify,X_val_cls_verify]:
    constant_cols = [cols for cols in col.columns 
                    if col[cols].nunique() == 1]
    print("Constant columns in:", constant_cols if constant_cols else "None ✅")

X_train_cls shape:  (27577, 17)
X_test_cls shape:  (2444, 17)
X_val_cls shape:  (2444, 17)

X_train_cls missing values:  0
X_test_cls missing values:  0
X_val_cls missing values:  0

X_train_cls categorical columns:  []
X_test_cls categorical columns:  []
X_val_cls categorical columns:  []

X_train_cls data types:
 person_age                        float64
person_income                     float64
person_emp_length                 float64
loan_grade                        float64
loan_amnt                         float64
loan_int_rate                     float64
loan_percent_income               float64
cb_person_default_on_file           int64
person_home_ownership_MORTGAGE    float64
person_home_ownership_OWN         float64
person_home_ownership_RENT        float64
loan_intent_DEBTCONSOLIDATION     float64
loan_intent_EDUCATION             float64
loan_intent_HOMEIMPROVEMENT       float64
loan_intent_MEDICAL               float64
loan_intent_PERSONAL              float64
loan_intent

In [43]:
# Regression   
print("X_train_reg shape: ",X_train_reg_verify.shape)
print("X_test_reg shape: ",X_test_reg_verify.shape)
print("X_val_reg shape: ",X_val_reg_verify.shape)

print()

print("X_train_reg missing values: ",X_train_reg_verify.isnull().sum().sum())
print("X_test_reg missing values: ",X_test_reg_verify.isnull().sum().sum())
print("X_val_reg missing values: ",X_val_reg_verify.isnull().sum().sum())

print()

print("X_train_reg categorical columns: ",X_train_reg_verify.select_dtypes(include=['object']).columns.tolist())
print("X_test_reg categorical columns: ",X_test_reg_verify.select_dtypes(include=['object']).columns.tolist())
print("X_val_reg categorical columns: ",X_val_reg_verify.select_dtypes(include=['object']).columns.tolist())

print()

print("X_train_reg data types:\n",X_train_reg_verify.dtypes)
print("X_test_reg data types:\n",X_test_reg_verify.dtypes)
print("X_val_reg data types:\n",X_val_reg_verify.dtypes)

print()

# Check 1: Infinite values
print("Infinite values in X_train_reg:", np.isinf(X_train_reg_verify).sum().sum())
print("Infinite values in X_test_reg:", np.isinf(X_test_reg_verify).sum().sum())
print("Infinite values in X_val_reg:", np.isinf(X_val_reg_verify).sum().sum())

print()
# Check 2: Constant columns
for col in [X_train_reg_verify,X_test_reg_verify,X_val_reg_verify]:
    constant_cols = [cols for cols in col.columns 
                    if col[cols].nunique() == 1]
    print("Constant columns in:", constant_cols if constant_cols else "None ✅")

X_train_reg shape:  (24943, 17)
X_test_reg shape:  (2210, 17)
X_val_reg shape:  (2210, 17)

X_train_reg missing values:  0
X_test_reg missing values:  0
X_val_reg missing values:  0

X_train_reg categorical columns:  []
X_test_reg categorical columns:  []
X_val_reg categorical columns:  []

X_train_reg data types:
 person_age                        float64
person_income                     float64
person_emp_length                 float64
loan_grade                        float64
loan_amnt                         float64
loan_status                         int64
loan_percent_income               float64
cb_person_default_on_file           int64
person_home_ownership_MORTGAGE    float64
person_home_ownership_OWN         float64
person_home_ownership_RENT        float64
loan_intent_DEBTCONSOLIDATION     float64
loan_intent_EDUCATION             float64
loan_intent_HOMEIMPROVEMENT       float64
loan_intent_MEDICAL               float64
loan_intent_PERSONAL              float64
loan_intent

### Step 12: Save Preprocessed Splits

In [37]:
X_train_cls_processed = X_train_cls_verify.copy()
X_test_cls_processed = X_test_cls_verify.copy()
X_val_cls_processed = X_val_cls_verify.copy()

X_train_reg_processed = X_train_reg_verify.copy()
X_test_reg_processed = X_test_reg_verify.copy()
X_val_reg_processed = X_val_reg_verify.copy()

In [38]:
# Load config
with open(config_file, "r") as f:
    config = yaml.safe_load(f)

    root_dir = os.path.dirname(os.path.dirname(os.path.abspath(config_file)))
    processed_path = os.path.join(root_dir, config['paths']['processed_data'])

# Create folder if it doesn't exist
os.makedirs(processed_path, exist_ok=True)

In [39]:
# Helper function to save splits
def save_processd(X, y, prefix, task_type):
    X.to_csv(os.path.join(processed_path, f"X_{prefix}_{task_type}.csv"), index=False)
    y.to_csv(os.path.join(processed_path, f"y_{prefix}_{task_type}.csv"), index=False)


In [42]:
# Save Classification processed
save_processd(X_train_cls_processed, y_train_cls_dup, "train", "cls_processed")
save_processd(X_val_cls_processed,   y_val_cls,   "val",   "cls_processed")
save_processd(X_test_cls_processed,  y_test_cls,  "test",  "cls_processed")

In [43]:
# Save Regression Splits
save_processd(X_train_reg_processed, y_train_reg_dup, "train", "reg_processed")
save_processd(X_val_reg_processed,   y_val_reg,   "val",   "reg_processed")
save_processd(X_test_reg_processed,  y_test_reg,  "test",  "reg_processed")

###  Step 13: Save Preprocessing Pipeline

In [44]:
pipeline = {
    'simple_imputer_cls':        simple_imputer_cls,
    'simple_imputer_reg':        simple_imputer_reg,
    'iqr_bounds_cls':     Train_cls_bounds,
    'iqr_bounds_reg':     Train_reg_bounds,
    'ordinal_encoder_cls': ordinal_encoder_cls,
    'ordinal_encoder_reg': ordinal_encoder_reg,
    'ohe_cls':            ohe_cls,
    'ohe_reg':            ohe_reg,
    'scaler_cls':         scaler_cls,
    'scaler_reg':         scaler_reg,
}

joblib.dump(pipeline, '../models/preprocessing_pipeline.joblib')
print("✅ Preprocessing pipeline saved successfully")

✅ Preprocessing pipeline saved successfully


### Step 14: Document Preprocessing Report

In [ ]:
# go and see docs/preprocessing_report.md